# ESM-2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wyqmath/ConformEval/blob/main/examples/notebooks/ESM-2.ipynb) · [GitHub](https://github.com/wyqmath/ConformEval/blob/main/examples/notebooks/ESM-2.ipynb)

Download benchmark data, run the real ESM-2 8M encoder, train grouped readers,
and independently score their predictions. **Run all cells in order.** The
default uses the complete knot-presence task: 569 unique sequences, 734 samples,
283 binding groups, and fixed 5 × 3 outer/inner folds. Set `FULL_BENCHMARK=False`
for a 20-sequence synthetic knot interface example.

Python >=3.9 and an Internet connection are required. CPU float32 is the
default; no GPU, API key or Google Drive access is needed. Data/code are fetched
from an immutable GitHub commit and checked against its blob hashes. Model
weights use an immutable Hugging Face revision. Each execution uses a fresh
workspace and preserves its actual environment, configuration and predictions.

## 1. Choose a task size and create a fresh workspace

The complete task reports a retrospective grouped-development score. The small
mode checks the software interface with synthetic labels. Frozen outer test
folds are used only for scoring; inner validation selects layers and reader
hyperparameters. Re-running from this cell creates a new run rather than
replacing previous outputs.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor
import csv, gzip, hashlib, json, math, os, platform, subprocess, sys, tempfile, time
import urllib.request

assert sys.version_info >= (3, 9), "Python >=3.9 is required"
FULL_BENCHMARK = True
SOURCE_REF = "a31fb65105e63087442a7351565f625bf2bbf820"
REPOSITORY = "wyqmath/ConformEval"
WORK_ROOT = Path(tempfile.mkdtemp(prefix="conformeval-notebook-"))
SOURCE_ROOT = WORK_ROOT / "repository"
SOURCE_ROOT.mkdir()
RUN_ROOT = WORK_ROOT / "runs"
RUN_ROOT.mkdir()
MODEL_CACHE = Path(os.environ.get("CONFORMEVAL_MODEL_CACHE", str(Path.home() / ".cache/huggingface/hub")))
STARTED_AT = datetime.now(timezone.utc).isoformat()
print("Mode:", "complete knot-presence benchmark" if FULL_BENCHMARK else "synthetic interface example")
print("Source commit:", SOURCE_REF)

## 2. Download code and task data

Only code, configuration and the selected task bundle are downloaded. This
includes about 20 MB of task data
in complete mode. The existing dataset manifest additionally checks scientific
file sizes and SHA-256 hashes before training or scoring.

In [ ]:
def fetch(url):
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "ConformEval-notebook"})
            with urllib.request.urlopen(request, timeout=60) as response:
                return response.read()
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)

tree_url = f"https://api.github.com/repos/{REPOSITORY}/git/trees/{SOURCE_REF}?recursive=1"
tree = json.loads(fetch(tree_url))
assert not tree.get("truncated"), "GitHub returned a partial file tree"
data_prefix = "data/base/" if FULL_BENCHMARK else "examples/"
root_files = {"pyproject.toml", "README.md", "LICENSE"}
selected = [entry for entry in tree["tree"] if entry["type"] == "blob" and
            (entry["path"] in root_files or entry["path"].startswith(("src/", "configs/", data_prefix)))]
assert selected

def download_entry(entry):
    relative = Path(entry["path"])
    assert not relative.is_absolute() and ".." not in relative.parts
    content = fetch(f"https://raw.githubusercontent.com/{REPOSITORY}/{SOURCE_REF}/{relative.as_posix()}")
    blob = hashlib.sha1(b"blob " + str(len(content)).encode() + b"\0" + content).hexdigest()
    assert blob == entry["sha"], f"Git blob mismatch: {relative}"
    destination = SOURCE_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(content)
    return {"path": relative.as_posix(), "bytes": len(content), "sha256": hashlib.sha256(content).hexdigest()}

with ThreadPoolExecutor(max_workers=4) as executor:
    downloaded = list(executor.map(download_entry, selected))
(RUN_ROOT / "download_manifest.json").write_text(
    json.dumps({"source_commit": SOURCE_REF, "files": downloaded}, indent=2), encoding="utf-8")
print(f"Downloaded and verified {len(downloaded)} files ({sum(x['bytes'] for x in downloaded):,} bytes)")

## 3. Install compatible dependencies

Install the scoring package and its ESM-2 extra. Pip chooses versions compatible
with this Python/runtime. Exact installed versions are recorded at the end.
On Colab, use the standard CPU runtime for the validated path.

In [ ]:
def run_step(name, arguments, cwd=SOURCE_ROOT):
    started = time.monotonic()
    completed = subprocess.run([str(arg) for arg in arguments], cwd=cwd,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    (RUN_ROOT / f"{name}.log").write_text(completed.stdout, encoding="utf-8")
    if completed.returncode:
        print(completed.stdout[-12000:])
        raise RuntimeError(f"{name} failed with exit code {completed.returncode}")
    print(f"{name}: PASS ({time.monotonic() - started:.2f} s)")

run_step("install", [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", ".[esm2]"])

## 4. Prepare exact sequences and download the pinned checkpoint

Input preparation deduplicates encoder inputs by sequence identity while keeping
all sample identities in training/scoring. The encoder uses ESM-2 layers 3/6
and 320-dimensional representations. The download command reuses a validated
local checkpoint cache when available.

In [ ]:
TASK = "KNOT-PRESENCE"
DATA_DIR = SOURCE_ROOT / ("data/base" if FULL_BENCHMARK else "examples/base")
MODEL_CONFIG = SOURCE_ROOT / "configs/models/esm2_8m.json"
PROTOCOL = SOURCE_ROOT / "configs/scoring.json"
example_flags = [] if FULL_BENCHMARK else ["--example"]

def cli(module, *arguments):
    return [sys.executable, "-m", "conformeval." + module, *map(str, arguments)]

run_step("prepare", cli("tasks", "--task", TASK, "--data-dir", DATA_DIR,
                        "--output-dir", RUN_ROOT / "inputs", *example_flags))
# The adapters module exposes download_main as a console entry point.
run_step("checkpoint", [sys.executable, "-c", "from conformeval.adapters import download_main; download_main()",
                         "--model-config", MODEL_CONFIG, "--cache-dir", MODEL_CACHE])
expected_sequences = 569 if FULL_BENCHMARK else 20
prepared = json.loads((RUN_ROOT / "inputs/run.json").read_text())
# Mini knot preparation selects 20 of the shared 30 example sequences.
assert prepared["unique_sequences"] == expected_sequences
print("Unique encoder inputs:", prepared["unique_sequences"])

## 5. Run real model inference

The sequence-only output retains both requested layers with mean pooling.
Special tokens are excluded. Sequences longer than 1,022 residues use windows
with 128-residue overlap; all original coordinates are retained. Extraction
runs without task labels.

In [ ]:
run_step("inference", cli("extract", "--model-config", MODEL_CONFIG,
                          "--input-fasta", RUN_ROOT / "inputs/sequences.fasta",
                          "--output-dir", RUN_ROOT / "features", "--cache-dir", MODEL_CACHE,
                          "--local-files-only", "--sequence-only", "--device", "cpu", "--threads", "4"))
manifest_path = RUN_ROOT / "features/representation_manifest.json"
features = json.loads(manifest_path.read_text())
assert features["output_dimension"] == 320
assert features["candidate_layers"] == [3, 6]
assert len(features["archives"]) == expected_sequences
print("Representations:", len(features["archives"]), "sequences; dimension", features["output_dimension"])

## 6. Train and validate readers

Each outer fold fits its scaler and reader on training groups only. Inner
validation chooses among layers 3/6 and C values 0.1/1.0. The final out-of-fold
predictions contain 734 samples per arm in full mode. Length, amino-acid
composition, nearest-neighbor and training-prior baselines use the same folds.

In [ ]:
run_step("training", cli("train", "--task", TASK, "--data-dir", DATA_DIR,
                         "--feature-manifest", manifest_path,
                         "--config", SOURCE_ROOT / "configs/readers/linear.json",
                         "--protocol", PROTOCOL, "--output-dir", RUN_ROOT / "readers", *example_flags))
training_metrics = json.loads((RUN_ROOT / "readers/metrics.json").read_text())
with gzip.open(RUN_ROOT / "readers/predictions.tsv.gz", "rt", encoding="utf-8", newline="") as handle:
    predictions = list(csv.DictReader(handle, delimiter="\t"))
nested = [row for row in predictions if row["arm"] == "PLM-nested"]
expected_samples = 734 if FULL_BENCHMARK else 20
assert len(nested) == expected_samples
assert len({row["sample_id"] for row in nested}) == expected_samples
assert len({row["outer_fold"] for row in nested}) == 5
for arm, metrics in sorted(training_metrics.items()):
    print(f"{arm:22s} AUROC {metrics['auroc']:.6f}")

## 7. Export one arm and score it independently

The public submission format is a four-column TSV. The scorer checks complete
sample identity, finite scores, masks and frozen group assignments. Its AUROC
and binding-group bootstrap interval must agree with the training output.

In [ ]:
submission = RUN_ROOT / "nested_predictions.tsv"
fields = ["task_id", "sample_id", "outer_fold", "score"]
with submission.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields, delimiter="\t", lineterminator="\n")
    writer.writeheader()
    writer.writerows({field: row[field] for field in fields} for row in nested)
run_step("rescoring", cli("evaluate_predictions", "--task", TASK, "--data-dir", DATA_DIR,
                          "--predictions", submission,
                          "--model-metadata", RUN_ROOT / "readers/model_metadata.json",
                          "--protocol", PROTOCOL, "--output-dir", RUN_ROOT / "rescored"))
rescored = json.loads((RUN_ROOT / "rescored/metrics.json").read_text())["metrics"][TASK]
original = training_metrics["PLM-nested"]
assert rescored["n_samples"] == expected_samples
assert math.isclose(rescored["auroc"], original["auroc"], rel_tol=0, abs_tol=1e-12)
assert rescored["group_bootstrap"]["ci95"] == original["group_bootstrap"]["ci95"]
print("Independent rescoring: PASS")
print("Nested-reader AUROC:", round(rescored["auroc"], 6))
print("95% binding-group bootstrap interval:", rescored["group_bootstrap"]["ci95"])

## 8. Save a portable verification record

`runs/` contains the downloaded-file manifest, prepared inputs, representation
archives, fitted readers, predictions, selection records, independent scores
and step logs. The compact record below links them by SHA-256. Its score is
reported with this run's environment; a synthetic run must never replace a
biological benchmark result. Files are temporary in Colab, so save required
outputs before ending the runtime.

In [ ]:
from importlib.metadata import version

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

record = {
    "status": "PASS", "project": "ConformEval", "source_commit": SOURCE_REF,
    "mode": "complete_knot_benchmark" if FULL_BENCHMARK else "synthetic_interface_example",
    "distribution": "conformeval", "package_version": version("conformeval"),
    "task": TASK, "model_id": features["model_id"], "model_revision": features["model_revision"],
    "started_at": STARTED_AT, "ended_at": datetime.now(timezone.utc).isoformat(),
    "platform": platform.platform(), "python": platform.python_version(), "device": "cpu",
    "dependencies": {name: version(name) for name in
                     ["numpy", "scipy", "scikit-learn", "torch", "transformers", "huggingface-hub"]},
    "unique_sequences": expected_sequences, "scored_samples": expected_samples,
    "groups": rescored["n_binding_groups"], "auroc": rescored["auroc"],
    "ci95": rescored["group_bootstrap"]["ci95"], "independent_rescoring_matches": True,
    "outputs_sha256": {path.relative_to(RUN_ROOT).as_posix(): sha256_file(path) for path in
        [RUN_ROOT / "download_manifest.json", manifest_path, RUN_ROOT / "readers/run.json",
         RUN_ROOT / "readers/predictions.tsv.gz", submission, RUN_ROOT / "rescored/metrics.json"]},
}
(RUN_ROOT / "notebook_verification.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print(json.dumps(record, indent=2))
print("Outputs:", RUN_ROOT)